# QLoRA fine-tune of Qwen2.5-7B-Instruct

Runs on a Kaggle GPU kernel. The notebook only orchestrates; the training code lives in
`src/` in the repository and reaches the kernel through the `chatdoctor-qlora-code`
dataset.

Settings the kernel needs: GPU accelerator on, internet on (the base weights come from
the Hugging Face Hub). Both live in `kernel-metadata.json`, so `kaggle kernels push`
carries them.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

T4 is compute capability 7.5. That rules out bf16 and flash-attention, which is why
the training script runs fp16 with the sdpa attention backend. Kaggle hands out two
cards; the run deliberately uses one, since a LoRA on a 7B model fits in 15 GB and a
single-device run keeps the step count and the loss curve easy to reason about.

In [ ]:
!pip install -q -U "transformers>=4.56,<6" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=0.34" "wandb>=0.17"
import torch, transformers, peft, bitsandbytes, wandb
print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__, wandb.__version__)

## Stage the code

Kaggle mounts a dataset read-only and flattens it, so the files are copied back into the
`src/` and `data/` layout the scripts expect.

The assertions are not decoration. A dataset that is still processing when the kernel
starts mounts as nothing at all, and since `!python` on a missing file does not raise,
the run would otherwise sail past the training cells and only fail at the plot, having
burned a GPU session for nothing. That is exactly how the first attempt died.

In [ ]:
import shutil, pathlib

root = pathlib.Path("/kaggle/input")
files = sorted(q for q in root.rglob("*") if q.is_file())
print(f"{len(files)} files under {root}")
for q in files[:40]:
    print("  ", q)

# Kaggle has moved this mount point between platform versions, so the code is located
# by filename instead of by an assumed path.
hits = sorted(root.rglob("train_lora.py"))
assert hits, "code dataset is not mounted, see the listing above"
SRC = hits[0].parent
print("using", SRC)

py, jsonl = sorted(SRC.glob("*.py")), sorted(SRC.glob("*.jsonl"))
assert len(py) >= 3 and len(jsonl) == 3, f"unexpected mount contents: {py} {jsonl}"

pathlib.Path("src").mkdir(exist_ok=True)
pathlib.Path("data").mkdir(exist_ok=True)
for f in py:
    shutil.copy(f, "src")
for f in jsonl:
    shutil.copy(f, "data")

assert pathlib.Path("src/train_lora.py").is_file()
!wc -l data/*.jsonl

## Smoke run

Thirty examples, one epoch. Catches an out-of-memory error or a bad argument in a couple
of minutes instead of forty. Checkpoints go to `/tmp` so they never reach the kernel
output, and logging is off so the smoke run does not show up as a W&B run of its own.

In [ ]:
!PYTHONPATH=src python src/train_lora.py \
    --limit-train 30 --limit-val 8 --epochs 1 --eval-steps 5 --save-steps 1000 \
    --out /tmp/smoke --no-wandb

## Full run

1000 examples, 3 epochs, effective batch 8, roughly 375 optimiser steps.

W&B runs in offline mode. Sending metrics live would mean putting an API key into the
kernel, and the only key-free way to do that on Kaggle is a secret created through the
web UI. Offline logging writes the same event stream to `/kaggle/working/wandb`, which
comes down with the rest of the output, and a `wandb sync` from a laptop turns it into a
normal run with a normal URL.

If the smoke run hit an out-of-memory error, add `--batch-size 1 --grad-accum 8`.

In [ ]:
!WANDB_MODE=offline WANDB_DIR=/kaggle/working PYTHONPATH=src python src/train_lora.py \
    --out /kaggle/working/adapter \
    --epochs 3 \
    --run-name qwen2.5-7b-chatdoctor-r16

## Loss curve

Drawn from `log_history.json` rather than from W&B, so the repository carries the plot
whether or not the run gets synced.

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("/kaggle/working/adapter")
history = json.loads((OUT / "log_history.json").read_text())
train = [(h["step"], h["loss"]) for h in history if "loss" in h]
evals = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(*zip(*train), lw=1, alpha=.7, label="train")
if evals:
    ax.plot(*zip(*evals), marker="o", ms=4, label="validation")
ax.set_xlabel("step"); ax.set_ylabel("loss")
ax.set_title("Qwen2.5-7B-Instruct, QLoRA r=16")
ax.legend(); ax.grid(alpha=.3)
Path("/kaggle/working/reports").mkdir(exist_ok=True)
fig.tight_layout(); fig.savefig("/kaggle/working/reports/loss_curve.png", dpi=150)
print(f"final train {train[-1][1]:.4f}" + (f", final eval {evals[-1][1]:.4f}" if evals else ""))

## What gets handed back

Adapter, loss curve, and the offline W&B directory. Intermediate checkpoints are dropped
so the output stays small enough to pull over the API; the 15 GB of base weights never
leave the Hub, which is the whole point of LoRA.

In [ ]:
!rm -rf /kaggle/working/adapter/checkpoints
!du -sh /kaggle/working/adapter /kaggle/working/wandb
!ls -la /kaggle/working/adapter